# Lung cancer survival prediction — desktop GUI

Run the final model-saving cell in `Model_building.ipynb` first. Then run the code cell below. It opens a separate desktop window with the patient form and prediction button.

In [8]:
import tkinter as tk
from tkinter import ttk, messagebox
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

model_path = Path('../data/best_survival_model.joblib')
if not model_path.exists():
    model_path = Path('data/best_survival_model.joblib')
bundle = joblib.load(model_path)
model = bundle['model']
model_name = bundle['model_name']
input_columns = bundle['input_columns']

data_path = Path('../data/lung_cancer_dataset_cleaned.csv')
if not data_path.exists():
    data_path = Path('data/lung_cancer_dataset_cleaned.csv')
reference_data = pd.read_csv(data_path)

MODEL_PURPOSE = ('Purpose: estimate the probability that a lung-cancer patient survives using information from a medical examination.')
binary_columns = {'Secondhand_Smoke', 'Family_History', 'Occupational_Hazard', 'Chronic_Lung_Disease', 'Asbestos_Exposure', 'Radon_Exposure', 'Previous_Cancer_History', 'Coughing', 'Shortness_of_Breath', 'Chest_Pain', 'Coughing_Blood', 'Weight_Loss', 'Wheezing', 'Recurrent_Infections', 'Swallowing_Difficulty', 'Finger_Clubbing'}

## Start the separate application window

In [9]:
def default_numeric_value(column):
    if column == 'Diagnosis_Date_Year':
        return float(pd.to_datetime(reference_data['Diagnosis_Date']).dt.year.median())
    if column == 'Diagnosis_Date_Month':
        return float(pd.to_datetime(reference_data['Diagnosis_Date']).dt.month.median())
    return float(pd.to_numeric(reference_data[column], errors='coerce').median())

def explain_prediction(patient, probability):
    reasons = []
    stage = str(patient.get('Cancer_Stage', '')).lower()
    if 'stage iv' in stage:
        reasons.append('Stage IV is an advanced cancer stage')
    elif 'stage iii' in stage:
        reasons.append('Stage III is an advanced cancer stage')
    elif 'stage i' in stage:
        reasons.append('Stage I is an earlier cancer stage')
    age = float(patient.get('Age', 0) or 0)
    if age >= 70:
        reasons.append('the patient is in a higher age group')
    symptom_count = sum(int(patient.get(c, 0) or 0) for c in binary_columns)
    if symptom_count >= 6:
        reasons.append('several symptoms were reported')
    if not reasons:
        reasons.append('the combined pattern of the entered medical features')
    level = 'higher' if probability >= 0.5 else 'lower'
    return f'This {level} estimate is associated with ' + ', '.join(reasons) + '.'

def launch_gui():
    root = tk.Tk()
    root.title('Lung Cancer Survival Prediction')
    root.geometry('760x720')
    root.minsize(650, 550)
    ttk.Label(root, text='Lung Cancer Survival Prediction', font=('Segoe UI', 18, 'bold')).pack(pady=(14, 2))
    ttk.Label(root, text=f'Model used: {model_name}', font=('Segoe UI', 10, 'italic')).pack()
    ttk.Label(root, text=MODEL_PURPOSE, wraplength=700, justify='center').pack(pady=(4, 12))

    outer = ttk.Frame(root)
    outer.pack(fill='both', expand=True, padx=14)
    canvas = tk.Canvas(outer, highlightthickness=0)
    scrollbar = ttk.Scrollbar(outer, orient='vertical', command=canvas.yview)
    form = ttk.Frame(canvas)
    canvas_window = canvas.create_window((0, 0), window=form, anchor='nw')
    canvas.configure(yscrollcommand=scrollbar.set)
    canvas.pack(side='left', fill='both', expand=True)
    scrollbar.pack(side='right', fill='y')

    controls = {}
    for row, column in enumerate(input_columns):
        ttk.Label(form, text=column + ':').grid(row=row, column=0, sticky='w', padx=8, pady=4)
        if column in binary_columns:
            variable = tk.StringVar(value='0')
            control = ttk.Combobox(form, textvariable=variable, values=['0', '1'], state='readonly', width=24)
        elif column in reference_data.columns and reference_data[column].dtype == 'object':
            options = sorted(reference_data[column].dropna().astype(str).unique().tolist())
            variable = tk.StringVar(value=options[0])
            control = ttk.Combobox(form, textvariable=variable, values=options, state='readonly', width=24)
        else:
            variable = tk.StringVar(value=str(round(default_numeric_value(column), 2)))
            control = ttk.Entry(form, textvariable=variable, width=27)
        control.grid(row=row, column=1, sticky='ew', padx=8, pady=4)
        controls[column] = variable
    form.columnconfigure(1, weight=1)

    def resize_canvas(event):
        canvas.itemconfigure(canvas_window, width=event.width)
        canvas.configure(scrollregion=canvas.bbox('all'))
    form.bind('<Configure>', resize_canvas)
    canvas.bind('<Configure>', resize_canvas)

    result = ttk.Label(root, text='Enter patient information and press Predict.', wraplength=700, justify='center')
    result.pack(pady=10)

    def predict():
        try:
            values = {}
            for column in input_columns:
                value = controls[column].get()
                if column in binary_columns:
                    values[column] = int(value)
                elif column in reference_data.columns and reference_data[column].dtype == 'object':
                    values[column] = value
                else:
                    values[column] = float(value)
            patient = pd.DataFrame([values], columns=input_columns)
            probability = float(model.predict_proba(patient)[0, 1])
            result.configure(text=f'Estimated survival chance: {probability:.1%}\n\n{explain_prediction(values, probability)}')
        except Exception as error:
            messagebox.showerror('Input error', f'Please check the entered values.\n\n{error}')

    ttk.Button(root, text='Predict survival chance', command=predict).pack(pady=(0, 14), ipadx=12, ipady=5)
    root.mainloop()

launch_gui()